# Lab 06: Automate Model Training with GitHub Actions

> **Source:** Microsoft Learning -- [mslearn-mlops](https://github.com/MicrosoftLearning/mslearn-mlops), `docs/06-automate-model-training.md`, `.github/workflows/*`

## Objectives

By the end of this lab you will be able to:

1. **Create** a service principal for non-interactive Azure authentication
2. **Configure** GitHub Secrets and Variables for CI/CD workflows
3. **Build** a manually triggered (`workflow_dispatch`) GitHub Actions workflow
4. **Build** a PR-triggered workflow with path filters
5. **Set** branch protection rules to enforce training checks before merge

| Estimated Time | Estimated Cost |
|---|---|
| ~25 minutes | ~$0.50 |

## Architecture Overview

```mermaid
graph LR
    DEV[Developer] -->|push/PR| GH[GitHub]
    GH -->|triggers| WF[GitHub Actions Workflow]
    WF -->|authenticates via| SP[Service Principal]
    SP -->|submits job to| AML[Azure ML Workspace]
    AML -->|trains on| CC[Compute Cluster]
```

GitHub Actions provides the CI/CD engine for MLOps. A developer pushes code or opens a pull request, which triggers an automated workflow. The workflow authenticates to Azure using a service principal, then submits a training job to an Azure ML workspace. This ensures every code change is validated with a full training run before it can be merged.

## Step 1: Create a Service Principal

A **service principal** is a non-interactive identity that GitHub Actions uses to authenticate with Azure. Unlike a user account, it can be scoped to specific resources and does not require MFA.

### Create the service principal

```bash
az ad sp create-for-rbac --name "ai300-lab-sp" --role contributor \
  --scopes /subscriptions/<sub-id>/resourceGroups/<rg-name> --sdk-auth
```

This command outputs a JSON object containing `clientId`, `clientSecret`, `subscriptionId`, and `tenantId`. You will store this entire JSON block as a GitHub secret.

**Key flags:**
- `--role contributor` -- grants read/write access to Azure resources (but not role assignments)
- `--scopes` -- limits the principal to a single resource group
- `--sdk-auth` -- formats the output for use with the `azure/login` GitHub Action

> **Exam Tip:** Service principals should always have **minimal permissions** scoped to specific resource groups. Never grant subscription-wide Contributor access for CI/CD pipelines. The principle of least privilege is a recurring exam theme.

## Step 2: Configure GitHub Secrets

After creating the service principal, you need to store the credentials in GitHub so that workflows can access them securely.

### Required secrets and variables

| Name | Type | Value |
|---|---|---|
| `AZURE_CREDENTIALS` | **Secret** | The full JSON output from `az ad sp create-for-rbac --sdk-auth` |
| `AZURE_RESOURCE_GROUP` | **Variable** | Your resource group name (e.g., `rg-ai300-dev`) |
| `AZURE_WORKSPACE_NAME` | **Variable** | Your workspace name (e.g., `mlw-ai300-dev`) |

### Where to configure

1. Navigate to your GitHub repository
2. Go to **Settings** > **Secrets and variables** > **Actions**
3. Click **New repository secret** to add `AZURE_CREDENTIALS`
4. Switch to the **Variables** tab and add `AZURE_RESOURCE_GROUP` and `AZURE_WORKSPACE_NAME`

> **Note:** Secrets are encrypted and never exposed in logs. Variables are plaintext but not sensitive -- they hold resource names, not credentials.

## Step 3: Manual Trigger Workflow

The simplest workflow uses `workflow_dispatch`, which adds a "Run workflow" button in the GitHub Actions UI. This is useful for ad-hoc training runs during development.

### `workflows/manual-trigger-job.yml`

```yaml
name: Manual trigger training job
on: workflow_dispatch
jobs:
  train:
    runs-on: ubuntu-latest
    steps:
      - name: Checkout
        uses: actions/checkout@v4
      - name: Azure Login
        uses: azure/login@v2
        with:
          creds: ${{ secrets.AZURE_CREDENTIALS }}
      - name: Submit ML job
        run: |
          az extension add -n ml
          az ml job create --file src/job.yml \
            --resource-group ${{ vars.AZURE_RESOURCE_GROUP }} \
            --workspace-name ${{ vars.AZURE_WORKSPACE_NAME }}
```

**Key concepts:**
- `workflow_dispatch` -- enables manual triggering from the Actions tab
- `azure/login@v2` -- authenticates using the service principal credentials stored in `AZURE_CREDENTIALS`
- `az extension add -n ml` -- installs the Azure ML CLI extension (not pre-installed on GitHub runners)
- `az ml job create` -- submits a training job defined in a YAML file to Azure ML

## Step 4: PR-Triggered Training

For production MLOps, you want training to run automatically whenever someone opens a pull request that modifies training code. The `pull_request` trigger with `paths` filters achieves this.

### `workflows/train-dev.yml`

```yaml
name: Train on PR to main
on:
  pull_request:
    branches: [main]
    paths: ['src/**']
jobs:
  train:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: azure/login@v2
        with:
          creds: ${{ secrets.AZURE_CREDENTIALS }}
      - name: Submit training job
        run: |
          az extension add -n ml
          az ml job create --file src/job.yml \
            --resource-group ${{ vars.AZURE_RESOURCE_GROUP }} \
            --workspace-name ${{ vars.AZURE_WORKSPACE_NAME }}
```

**Key differences from the manual workflow:**
- `pull_request` trigger fires automatically on PR creation/update
- `branches: [main]` -- only triggers on PRs targeting the main branch
- `paths: ['src/**']` -- only triggers when files under `src/` are changed

> **Exam Tip:** Path filters prevent workflows from running on unrelated changes (e.g., documentation updates, config changes). This saves compute costs and reduces noise.

## Step 5: Branch Protection

Branch protection rules ensure that code cannot be merged to `main` without passing all required checks. In an MLOps context, this means a model must train successfully before the code change is accepted.

### Configuring branch protection in GitHub

1. Go to **Settings** > **Branches**
2. Click **Add branch protection rule**
3. Set **Branch name pattern** to `main`
4. Enable:
   - **Require a pull request before merging**
   - **Require status checks to pass before merging**
   - Select the `train` job from the `Train on PR to main` workflow
5. Click **Create**

### What this achieves

- No direct pushes to `main` -- all changes go through pull requests
- The training workflow must complete successfully before the PR can be merged
- Reviewers can inspect training metrics before approving
- If training fails (e.g., accuracy drops), the PR is blocked automatically

## Feature Branch Workflow

The complete MLOps development cycle follows this pattern:

1. **Branch** -- create a feature branch from `main` (e.g., `feature/add-regularization`)
2. **Code** -- modify training scripts, hyperparameters, or data processing
3. **Push & PR** -- push the branch and open a pull request to `main`
4. **Auto-train** -- the `Train on PR to main` workflow triggers automatically
5. **Review** -- team reviews the training metrics (accuracy, loss, etc.)
6. **Merge** -- once checks pass and reviewers approve, merge to `main`

```
main:    ----*-----------*---- (protected)
              \         /
feature:       *---*---*        (training runs on each push)
```

> **Exam Tip:** This feature-branch-to-PR-to-merge cycle is the standard MLOps CI/CD pattern tested on the exam. Know the difference between `workflow_dispatch` (manual), `push` (on merge), and `pull_request` (on PR) triggers.

## Key Takeaways

1. **Service principals** -- non-interactive identities for CI/CD authentication; always scope to specific resource groups
2. **GitHub Secrets** -- store sensitive credentials (service principal JSON) encrypted; use Variables for non-sensitive config
3. **`workflow_dispatch`** -- manual trigger for ad-hoc training runs during development
4. **`pull_request` with path filters** -- automated trigger that only runs when relevant code changes, saving compute costs
5. **Branch protection** -- enforces that training must succeed before code can be merged to main

> **Next:** In Lab 07, you will deploy trained models to managed online endpoints and set up monitoring for data drift.